# DocFusion: Model Training & Anomaly Detection

This notebook documents the training pipeline for the XGBoost-based forgery detector.
It covers feature extraction (visual + statistical), model training, evaluation,
and feature importance analysis.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = str(Path(".").resolve().parent)
sys.path.insert(0, PROJECT_ROOT)

from ocr_engine import OCREngine
from field_extractor import FieldExtractor
from feature_extractor import FeatureExtractor
from anomaly_detector import AnomalyDetector
from utils import load_jsonl, load_image

## 1. Feature Extraction Pipeline

Each document image goes through two parallel feature pipelines:

**Visual features (10):**
- ELA (Error Level Analysis): mean, max, std, high_ratio — detects copy-paste edits
- Edge density (Canny) — unnatural edge patterns in forged regions
- Noise level (Laplacian variance) — inconsistent noise from splicing
- Brightness mean/std — lighting inconsistencies
- LBP uniformity/entropy — texture anomalies

**Statistical features (8):**
- Field completeness (vendor/date/total present?)
- Total value, z-score, is_round
- OCR confidence mean/min
- Text length, number of lines

In [ ]:
# Load training data
train_path = os.path.join(PROJECT_ROOT, "dummy_data", "train", "train.jsonl")
records = load_jsonl(train_path)
images_dir = os.path.join(PROJECT_ROOT, "dummy_data", "train", "images")

print(f"Training records: {len(records)}")
print(f"Forged: {sum(1 for r in records if r['label']['is_forged'])}")
print(f"Genuine: {sum(1 for r in records if not r['label']['is_forged'])}")

In [ ]:
# Extract features for all training images
ocr = OCREngine()
extractor = FieldExtractor()
feat_ext = FeatureExtractor()

features_list = []
labels = []

for i, rec in enumerate(records):
    img_name = Path(rec["image_path"]).name
    img_path = os.path.join(images_dir, img_name)
    image = load_image(img_path)

    if image is None:
        features_list.append(feat_ext.default_features())
        labels.append(rec["label"]["is_forged"])
        continue

    ocr_result = ocr.run(image)
    fields = extractor.extract(ocr_result)
    vis = feat_ext.visual_features(image)
    stat = feat_ext.statistical_features(fields, ocr_result)
    features_list.append({**vis, **stat})
    labels.append(rec["label"]["is_forged"])

    if (i + 1) % 5 == 0:
        print(f"  Processed {i + 1}/{len(records)}")

print(f"\nExtracted {len(features_list)} feature vectors with {len(features_list[0])} features each")

## 2. Feature Distribution Analysis

Visualize how features differ between genuine and forged documents.

In [ ]:
feature_names = sorted(features_list[0].keys())
X = np.array([[f.get(n, 0.0) for n in feature_names] for f in features_list])
y = np.array(labels)

genuine_mask = y == 0
forged_mask = y == 1

# Plot distributions for key features
key_features = ["ela_mean", "ela_high_ratio", "noise_level", "edge_density",
                "ocr_conf_mean", "field_completeness", "total_value", "lbp_entropy"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for ax, feat in zip(axes, key_features):
    idx = feature_names.index(feat) if feat in feature_names else None
    if idx is None:
        continue
    ax.hist(X[genuine_mask, idx], bins=10, alpha=0.6, label="Genuine", color="#4ADE80")
    ax.hist(X[forged_mask, idx], bins=10, alpha=0.6, label="Forged", color="#F87171")
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle("Feature Distributions: Genuine vs Forged", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Model Training

Train the XGBoost classifier and evaluate with cross-validation.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

detector = AnomalyDetector()
detector.fit(features_list, labels)

print("Model type:", type(detector.model).__name__)
print(f"Features: {len(feature_names)}")
print(f"Training samples: {len(labels)}")
print(f"Class distribution: genuine={sum(genuine_mask)}, forged={sum(forged_mask)}")

In [ ]:
# Cross-validation (if enough samples)
if len(labels) >= 6 and len(set(labels)) > 1:
    n_splits = min(5, min(sum(genuine_mask), sum(forged_mask)))
    if n_splits >= 2:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        scores = cross_val_score(detector.model, X, y, cv=cv, scoring="accuracy")
        print(f"Cross-validation accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")
        print(f"Per-fold scores: {[f'{s:.3f}' for s in scores]}")
    else:
        print("Not enough samples per class for cross-validation")
else:
    print("Insufficient data for cross-validation")

In [ ]:
# Training set predictions (in-sample)
preds = detector.predict_batch(features_list)
print("\n--- Training Set Performance (in-sample) ---")
print(classification_report(labels, preds, target_names=["Genuine", "Forged"], zero_division=0))

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Genuine", "Forged"])
ax.set_yticklabels(["Genuine", "Forged"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix (Training Set)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=16)
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 4. Feature Importance

Which features contribute most to the forgery detection decision?

In [ ]:
if detector.model is not None and hasattr(detector.model, "feature_importances_"):
    importances = detector.model.feature_importances_
    sorted_idx = np.argsort(importances)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh([feature_names[i] for i in sorted_idx], importances[sorted_idx], color="#6C9FFF")
    ax.set_xlabel("Importance")
    ax.set_title("Feature Importance (XGBoost)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print("\nTop 5 features:")
    for i in sorted_idx[-5:][::-1]:
        print(f"  {feature_names[i]:25s} {importances[i]:.4f}")
else:
    print("Model not trained or does not expose feature importances.")

## 5. Save Model

Save the trained model and feature statistics for use in the prediction pipeline.

In [ ]:
model_dir = os.path.join(PROJECT_ROOT, "tmp_work", "model")
os.makedirs(model_dir, exist_ok=True)

detector.save(model_dir)
feat_ext.save_stats(features_list, model_dir)

print(f"Model saved to: {model_dir}")
for f in os.listdir(model_dir):
    size = os.path.getsize(os.path.join(model_dir, f))
    print(f"  {f}: {size / 1024:.1f} KB")

## 6. Design Decisions

| Decision | Rationale |
|---|---|
| XGBoost over deep learning | CPU-only constraint, <1MB model, <500MB RAM |
| ELA for tampering detection | Standard forensic technique, no training needed |
| LBP for texture analysis | Detects spliced regions with different texture patterns |
| 100 estimators, depth 4 | Balance between accuracy and inference speed |
| 0.5 probability threshold | Default; could be tuned with validation data |
| Fallback to GradientBoosting | Handles environments where XGBoost isn't available |